# 00 · Revisión del entorno

Este cuaderno se corre una vez en cada máquina donde vaya a trabajar el proyecto
(el Mac con VS Code, y más adelante Colab). No procesa datos. Solo responde tres
preguntas: qué versiones hay instaladas, si PyTorch ve una GPU, y cuánta memoria y
disco quedan. Copia la salida de la última celda en la bitácora.

Por qué importa: el plan del proyecto reparte el trabajo en dos lugares. Todo lo que
sea leer DICOM, calcular SUV, remuestrear y evaluar corre en el Mac. El entrenamiento
con el presupuesto completo (25 000 iteraciones por modelo) corre en Colab, salvo que
el Mac tenga memoria de sobra y las pruebas cortas muestren que rinde. Esta celda es
la que decide.

In [ ]:
import sys, platform, importlib, shutil, os

print("Python", sys.version.split()[0], "en", platform.platform())
print("Procesador:", platform.machine())

paquetes = ["numpy", "scipy", "pandas", "pydicom", "SimpleITK", "nibabel",
            "skimage", "matplotlib", "yaml", "pytest", "torch", "monai", "tcia_utils"]
for nombre in paquetes:
    try:
        mod = importlib.import_module(nombre)
        print(f"  {nombre:12s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {nombre:12s} NO instalado")

## ¿Hay GPU?

En Colab la respuesta esperada es `cuda`. En un Mac con chip M la respuesta es `mps`,
que es el nombre que PyTorch le da al chip gráfico de Apple. Si dice `cpu`, el
entrenamiento igual funciona, solo que mucho más lento.

In [ ]:
import sys
sys.path.insert(0, "../src")
from petct.device import describe_device
print("Dispositivo que usaría PyTorch:", describe_device())

## Memoria y disco

Regla práctica para este proyecto: los 250 estudios en NIfTI a resolución nativa
ocupan unos 25 GB; a 3 mm, menos de 8 GB. Un parche 3D de 96³ con dos canales y lote
de 2 pesa poco, pero la U-Net guarda activaciones intermedias, y ahí se van varios
GB. Con 16 GB de memoria unificada en el Mac alcanza para probar; para el presupuesto
completo conviene Colab o un Mac con 32 GB o más.

In [ ]:
import shutil, os
total, usado, libre = shutil.disk_usage(os.getcwd())
print(f"Disco libre en esta carpeta: {libre / 2**30:.1f} GB de {total / 2**30:.1f} GB")
try:
    import psutil
    print(f"Memoria RAM total: {psutil.virtual_memory().total / 2**30:.1f} GB")
except ImportError:
    print("psutil no instalado; en el Mac: sysctl hw.memsize en la Terminal (bytes)")

## Qué anotar en la bitácora

Fecha, máquina, versión de Python, dispositivo (`cuda`, `mps` o `cpu`), RAM y disco
libre. Con eso, cualquiera que lea el informe sabe en qué condiciones se corrió cada
etapa, y tú sabes si el Paso 3 se hace aquí o en Colab.